In [3]:
import gzip
import json
import gc
from pathlib import Path
import duckdb

import pandas as pd
import numpy as np

In [12]:

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
TEMP_DIR = Path("../data/temp_reviews")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)
DEDUP_PATH = PROCESSED_DIR / "reviews_dedup.parquet"
FILTERED_PATH = PROCESSED_DIR / "reviews_filtered_min5.parquet"

REVIEWS_PATH = RAW_DIR / "Electronics.json.gz"
META_PATH = RAW_DIR / "meta_Electronics.json.gz"

In [5]:
def read_json_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

## STEP 1 — CLEAN REVIEW DATA

Mục tiêu:
- remove invalid rows
- normalize text
- convert timestamp
- remove bad interactions

Review cleaning pipeline

In [7]:
def clean_review_row(row):
    reviewer_id = row.get("reviewerID")
    asin = row.get("asin")
    overall = row.get("overall")
    review_text = row.get("reviewText", "")
    summary = row.get("summary", "")
    unix_time = row.get("unixReviewTime")
    verified = row.get("verified", False)
    vote = row.get("vote", 0)

    if reviewer_id is None or asin is None or overall is None:
        return None

    try:
        overall = float(overall)
    except:
        return None

    if overall < 1 or overall > 5:
        return None

    if not isinstance(review_text, str):
        review_text = ""

    if not isinstance(summary, str):
        summary = ""

    review_text = review_text.strip().lower()
    summary = summary.strip().lower()

    try:
        if isinstance(vote, str):
            vote = vote.replace(",", "")
        vote = int(vote)
    except:
        vote = 0

    try:
        timestamp = pd.to_datetime(unix_time, unit="s")
    except:
        return None

    return {
        "user_id": reviewer_id,
        "item_id": asin,
        "rating": overall,
        "review_text": review_text,
        "summary": summary,
        "verified": int(bool(verified)),
        "vote": vote,
        "helpful_score": np.log1p(vote),
        "timestamp": timestamp,
    }

In [4]:
def clean_reviews_to_chunks(path, out_dir, chunk_size=300_000, limit=None):
    buffer = []
    chunk_id = 0
    total_read = 0
    total_saved = 0

    for row in read_json_gz(path):
        total_read += 1

        cleaned = clean_review_row(row)

        if cleaned is not None:
            buffer.append(cleaned)

        if len(buffer) >= chunk_size:
            df = pd.DataFrame(buffer)

            out_path = out_dir / f"reviews_clean_chunk_{chunk_id:04d}.parquet"
            df.to_parquet(out_path, index=False)

            total_saved += len(df)

            print(
                f"Saved chunk {chunk_id:04d} | "
                f"rows: {len(df):,} | "
                f"total read: {total_read:,} | "
                f"total saved: {total_saved:,}"
            )

            buffer.clear()
            del df
            gc.collect()

            chunk_id += 1

        if limit is not None and total_read >= limit:
            break

    if buffer:
        df = pd.DataFrame(buffer)

        out_path = out_dir / f"reviews_clean_chunk_{chunk_id:04d}.parquet"
        df.to_parquet(out_path, index=False)

        total_saved += len(df)

        print(
            f"Saved final chunk {chunk_id:04d} | "
            f"rows: {len(df):,} | "
            f"total read: {total_read:,} | "
            f"total saved: {total_saved:,}"
        )

        buffer.clear()
        del df
        gc.collect()

    print("DONE")
    print("Total read:", total_read)
    print("Total saved:", total_saved)

In [8]:
# chạy thử với 1 000 000 dòng
clean_reviews_to_chunks(
    REVIEWS_PATH,
    TEMP_DIR,
    chunk_size=100_000,
    limit=1_000_000
)

Saved chunk 0000 | rows: 100,000 | total read: 100,000 | total saved: 100,000
Saved chunk 0001 | rows: 100,000 | total read: 200,000 | total saved: 200,000
Saved chunk 0002 | rows: 100,000 | total read: 300,000 | total saved: 300,000
Saved chunk 0003 | rows: 100,000 | total read: 400,000 | total saved: 400,000
Saved chunk 0004 | rows: 100,000 | total read: 500,000 | total saved: 500,000
Saved chunk 0005 | rows: 100,000 | total read: 600,000 | total saved: 600,000
Saved chunk 0006 | rows: 100,000 | total read: 700,000 | total saved: 700,000
Saved chunk 0007 | rows: 100,000 | total read: 800,000 | total saved: 800,000
Saved chunk 0008 | rows: 100,000 | total read: 900,000 | total saved: 900,000
Saved chunk 0009 | rows: 100,000 | total read: 1,000,000 | total saved: 1,000,000
DONE
Total read: 1000000
Total saved: 1000000


In [9]:
# nếu thấy chạy thử ổn thì xóa các file tạm 
for p in TEMP_DIR.glob("reviews_clean_chunk_*.parquet"):
    p.unlink()

In [10]:
# chạy lại với toàn bộ dữ liệu
clean_reviews_to_chunks(
    REVIEWS_PATH,
    TEMP_DIR,
    chunk_size=300_000,
    limit=None
)

Saved chunk 0000 | rows: 300,000 | total read: 300,000 | total saved: 300,000
Saved chunk 0001 | rows: 300,000 | total read: 600,000 | total saved: 600,000
Saved chunk 0002 | rows: 300,000 | total read: 900,000 | total saved: 900,000
Saved chunk 0003 | rows: 300,000 | total read: 1,200,000 | total saved: 1,200,000
Saved chunk 0004 | rows: 300,000 | total read: 1,500,000 | total saved: 1,500,000
Saved chunk 0005 | rows: 300,000 | total read: 1,800,000 | total saved: 1,800,000
Saved chunk 0006 | rows: 300,000 | total read: 2,100,000 | total saved: 2,100,000
Saved chunk 0007 | rows: 300,000 | total read: 2,400,000 | total saved: 2,400,000
Saved chunk 0008 | rows: 300,000 | total read: 2,700,000 | total saved: 2,700,000
Saved chunk 0009 | rows: 300,000 | total read: 3,000,000 | total saved: 3,000,000
Saved chunk 0010 | rows: 300,000 | total read: 3,300,000 | total saved: 3,300,000
Saved chunk 0011 | rows: 300,000 | total read: 3,600,000 | total saved: 3,600,000
Saved chunk 0012 | rows: 300

In [ ]:
# kiểm tra chunk đã tạo
from pathlib import Path

TEMP_DIR = Path("../data/temp_reviews")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

chunk_files = sorted(TEMP_DIR.glob("reviews_clean_chunk_*.parquet"))

print("Number of chunks:", len(chunk_files))
print("First:", chunk_files[0])
print("Last:", chunk_files[-1])

Number of chunks: 70
First: ..\data\temp_reviews\reviews_clean_chunk_0000.parquet
Last: ..\data\temp_reviews\reviews_clean_chunk_0069.parquet


In [7]:
# Dùng DuckDB để đọc nhiều parquet
con = duckdb.connect()
REVIEWS_CHUNKS = str(TEMP_DIR / "reviews_clean_chunk_*.parquet")
print(REVIEWS_CHUNKS)
con.execute(f"""
SELECT COUNT(*) AS n_rows
FROM read_parquet('{REVIEWS_CHUNKS}')
""").df()

..\data\temp_reviews\reviews_clean_chunk_*.parquet


,n_rows
0,20994353


## STEP2: Remove duplicate user-item

In [19]:
# giữ lại review mới nhất của mỗi cặp theo timestamp (user_id, item_id)
DEDUP_PATH = PROCESSED_DIR / "reviews_dedup.parquet"

con.execute(f"""
COPY (
    SELECT
        user_id,
        item_id,
        rating,
        review_text,
        summary,
        verified,
        vote,
        helpful_score,
        timestamp
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY user_id, item_id
                ORDER BY timestamp DESC
            ) AS rn
        FROM read_parquet('{REVIEWS_CHUNKS}')
    )
    WHERE rn = 1
) TO '{DEDUP_PATH}' (FORMAT PARQUET)
""")

In [20]:
# kiểm tra lại sau khi loại duplicate
con.execute(f"""
SELECT COUNT(*) AS n_rows
FROM read_parquet('{DEDUP_PATH}')
""").df()

,n_rows
0,20553480


In [21]:
# Đếm user/item sau duplicate
con.execute(f"""
SELECT
    COUNT(*) AS interactions,
    COUNT(DISTINCT user_id) AS users,
    COUNT(DISTINCT item_id) AS items
FROM read_parquet('{DEDUP_PATH}')
""").df()

,interactions,users,items
0,20553480,9838676,756489


## Step3 : Filter user >= 5 và item >= 5

In [22]:
FILTERED_PATH = PROCESSED_DIR / "reviews_filtered_min5.parquet"

MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5

con.execute(f"""
COPY (
    WITH user_counts AS (
        SELECT user_id, COUNT(*) AS user_n
        FROM read_parquet('{DEDUP_PATH}')
        GROUP BY user_id
    ),
    item_counts AS (
        SELECT item_id, COUNT(*) AS item_n
        FROM read_parquet('{DEDUP_PATH}')
        GROUP BY item_id
    )
    SELECT r.*
    FROM read_parquet('{DEDUP_PATH}') r
    JOIN user_counts u
        ON r.user_id = u.user_id
    JOIN item_counts i
        ON r.item_id = i.item_id
    WHERE u.user_n >= {MIN_USER_INTERACTIONS}
      AND i.item_n >= {MIN_ITEM_INTERACTIONS}
) TO '{FILTERED_PATH}' (FORMAT PARQUET)
""")

In [23]:
#Kiểm tra kết quả sau filter
filtered_stats = con.execute(f"""
SELECT
    COUNT(*) AS interactions,
    COUNT(DISTINCT user_id) AS users,
    COUNT(DISTINCT item_id) AS items,
    MIN(timestamp) AS min_time,
    MAX(timestamp) AS max_time
FROM read_parquet('{FILTERED_PATH}')
""").df()

filtered_stats

,interactions,users,items,min_time,max_time
0,7059313,788112,284114,1999-06-13,2018-10-04


In [24]:
## tính lại độ thưa (sparsity mới)
interactions = filtered_stats.loc[0, "interactions"]
users = filtered_stats.loc[0, "users"]
items = filtered_stats.loc[0, "items"]

sparsity = 1 - interactions / (users * items)

print("Interactions:", interactions)
print("Users:", users)
print("Items:", items)
print("Sparsity:", sparsity)
print("Sparsity %:", sparsity * 100)


Interactions: 7059313
Users: 788112
Items: 284114
Sparsity: 0.9999684730568559
Sparsity %: 99.99684730568559


In [25]:
# kiểm tra phân phối rating sau khi filter
rating_after_filter = con.execute(f"""
SELECT
    rating,
    COUNT(*) AS count,
    COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS percentage
FROM read_parquet('{FILTERED_PATH}')
GROUP BY rating
ORDER BY rating
""").df()

rating_after_filter

,rating,count,percentage
0,1.0,502382,7.116585
1,2.0,325826,4.615548
2,3.0,533457,7.556784
3,4.0,1186916,16.813477
4,5.0,4510732,63.897606


In [26]:
# Kiểm tra user/item distribution sau filter
user_dist = con.execute(f"""
SELECT user_n, COUNT(*) AS num_users
FROM (
    SELECT user_id, COUNT(*) AS user_n
    FROM read_parquet('{FILTERED_PATH}')
    GROUP BY user_id
)
GROUP BY user_n
ORDER BY user_n
LIMIT 20
""").df()

user_dist

,user_n,num_users
0,1,117
1,2,681
2,3,4888
3,4,36365
4,5,206739
5,6,133081
6,7,90242
7,8,64353
8,9,47410
9,10,35896


In [27]:
item_dist = con.execute(f"""
SELECT item_n, COUNT(*) AS num_items
FROM (
    SELECT item_id, COUNT(*) AS item_n
    FROM read_parquet('{FILTERED_PATH}')
    GROUP BY item_id
)
GROUP BY item_n
ORDER BY item_n
LIMIT 20
""").df()

item_dist

,item_n,num_items
0,1,27666
1,2,33652
2,3,31637
3,4,25942
4,5,19838
5,6,14913
6,7,11889
7,8,9914
8,9,8182
9,10,6995


## LƯU KẾT QUẢ PHASE 2

In [28]:
phase2_summary = filtered_stats.copy()
phase2_summary["sparsity"] = sparsity
phase2_summary["min_user_interactions"] = MIN_USER_INTERACTIONS
phase2_summary["min_item_interactions"] = MIN_ITEM_INTERACTIONS

phase2_summary.to_csv(PROCESSED_DIR / "phase2_filtered_summary.csv", index=False)

phase2_summary

,interactions,users,items,min_time,max_time,sparsity,min_user_interactions,min_item_interactions
0,7059313,788112,284114,1999-06-13,2018-10-04,0.999968,5,5


In [8]:
dup_before = con.execute(f"""
SELECT COUNT(*) AS duplicate_pairs_before
FROM (
    SELECT
        user_id,
        item_id,
        COUNT(*) AS cnt
    FROM read_parquet('{REVIEWS_CHUNKS}')
    GROUP BY user_id, item_id
    HAVING COUNT(*) > 1
)
""").df()

dup_before

,duplicate_pairs_before
0,435899


In [11]:
dup_after = con.execute(f"""
SELECT COUNT(*) AS duplicate_pairs_after
FROM (
    SELECT
        user_id,
        item_id,
        COUNT(*) AS cnt
    FROM read_parquet('{DEDUP_PATH}')
    GROUP BY user_id, item_id
    HAVING COUNT(*) > 1
)
""").df()

dup_after

,duplicate_pairs_after
0,0


In [13]:
dup_after_filtered = con.execute(f"""
SELECT COUNT(*) AS duplicate_pairs_after
FROM (
    SELECT
        user_id,
        item_id,
        COUNT(*) AS cnt
    FROM read_parquet('{FILTERED_PATH}')
    GROUP BY user_id, item_id
    HAVING COUNT(*) > 1
)
""").df()

dup_after_filtered

,duplicate_pairs_after
0,0
